# 手动运行：CT 数字岩心 -> pnextract 孔网 -> 分机制 SIP/AC3D 复现

这个 notebook 把一次完整的 Niu 2020 Berea 风格流程串起来：

1. 输入 CT 分割体和 SIP 参数。
2. 导出 Fiji/VTK 风格三维数字岩心 HTML。
3. 调用 pnextract 提取孔隙网络，导出球棍网络 HTML。
4. 导出孔节点半径和孔喉长度分布直方图。
5. 自动读取提取后的孔网几何参数，计算 pore/membrane 极化谱。
6. 按 `interfacial / pore / membrane / all` 生成 AC3D 输入谱。
7. 可选运行 full-grid GPU AC3D x/y/z sweep。
8. 汇总方向平均并导出 SIP 机制对比图。

默认路径指向这次结果包：`results/niu2020_berea_reproduction_20260625_solver_hardening`。如果换样品，只改第一个代码单元。

In [1]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# =========================
# 1. 用户可改参数区
# =========================
PROJECT_ROOT = Path(r"C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟")
PYTHON_EXE = Path(r"C:\Users\imgw\.conda\envs\ml\python.exe")

# 本次手动运行结果目录。默认写到这次已完成的 Niu 2020 结果包。
RUN_DIR = PROJECT_ROOT / "results" / "niu2020_berea_reproduction3"

# CT 输入：Fiji/pnextract/AC3D full-grid 求解统一使用同一套 TIFF 派生二值体。
SOURCE_SEGMENTED_TIFF = PROJECT_ROOT / "data" / "Niu 2020data" / "microCT_Berea.tiff"
AC3D_RAW = None
AC3D_SHAPE_ZYX = (350, 350, 350)
AC3D_PORE_LABEL = 0
AC3D_SOLID_LABEL = 255
VOXEL_SIZE_UM = 2.8
VOXEL_SIZE_M = VOXEL_SIZE_UM * 1e-6

# 将 SOURCE_SEGMENTED_TIFF 重映射为 pore=0, solid=255。
# Niu microCT_Berea.tiff/raw 的标签是 1=pore, 2=solid。
SOURCE_PORE_VALUES = [1]
BINARY_SOLID_VALUE = 255

# SIP/极化参数。默认是 Niu et al. (2020) Table 1 / Section 4.2。
SIP_PARAMETERS = {
    "surface_conductance_s": 1.3e-9,
    "diffusion_coefficient_m2_s": 1.3e-9,
    "dynamic_pore_size_m": 2.7e-6,
    "membrane_polarizability": 0.003,
    "water_conductivity_s_m": 0.043,
    "epsilon_0_f_m": 8.85e-12,
    "water_relative_permittivity": 80.0,
    "solid_relative_permittivity": 7.0,
}

# checkpoint 频点。正式完整扫频可自行加密，但先保留这次复现实测通过的 checkpoint 网格。
FREQUENCIES_HZ = [1e-3, 1e-2, 1e-1, 1.0, 1e3, 1e6, 1e9]
MECHANISMS = ["interfacial", "pore", "membrane", "all"]
DIRECTIONS = ["x", "y", "z"]

# 运行开关。full-grid 很重，默认不重跑；若要从 notebook 手动完整重跑，改为 True。
REMAP_BINARY_VOLUME = True
EXPORT_FIJI3D_HTML = True
RUN_PNEXTRACT_AND_RENDER_NETWORK = True
COMPUTE_POLARIZATION_SPECTRA = True
MAKE_COMPONENT_SPECTRA = True
RUN_FULLGRID_AC3D = True
SUMMARIZE_AND_PLOT = True

# pnextract 和 AC3D 数值设置。
PNEXTRACT_EXE = PROJECT_ROOT / "code" / "vendor" / "pnextract" / "bin" / "pnextract.exe"
PNEXTRACT_DOWNSAMPLE = 1
FIJI3D_DOWNSAMPLE = 1
DISTRIBUTION_BINS = 48

AC3D_DTYPE = "complex128"
AC3D_PRECONDITIONER = "fft"
AC3D_FFT_REFERENCE = "pore"
AC3D_GAUGE_MODE = "auto"
AC3D_RTOL = "1e-5"
AC3D_ATOL = "0"
AC3D_MAXITER = "1000"
AC3D_RESIDUAL_EVERY = "50"

# 可视化输出文件名按你的要求统一放在 RUN_DIR/pore_network 下。
PORE_NETWORK_DIR = RUN_DIR / "pore_network"
SEGMENTED_CORE_DIR = RUN_DIR / "segmented_core"
FORMAL_INPUT_DIR = RUN_DIR / "source_data" / "formal_inputs"
COMPONENT_SPECTRA_DIR = FORMAL_INPUT_DIR / "component_spectra_manual"
SIM_SWEEP_DIR = RUN_DIR / "simulation_sweeps"
FIGURE_DIR = RUN_DIR / "figures"
PROVENANCE_DIR = RUN_DIR / "provenance"

BINARY_TIFF = SEGMENTED_CORE_DIR / "microCT_Berea_solid255_pore0.tiff"
BINARY_RAW = SEGMENTED_CORE_DIR / "microCT_Berea_solid255_pore0.raw"
BINARY_METADATA = SEGMENTED_CORE_DIR / "microCT_Berea_solid255_pore0_remap_metadata.json"

FIJI3D_HTML = PORE_NETWORK_DIR / "microCT_Berea_fiji3d_fullres_volume_interactive.html"
FIJI3D_METADATA = PORE_NETWORK_DIR / "microCT_Berea_fiji3d_fullres_volume_interactive_metadata.json"
BALLSTICK_HTML = PORE_NETWORK_DIR / "microCT_Berea_pnextract_ballstick_interactive.html"
BALLSTICK_METADATA = PORE_NETWORK_DIR / "microCT_Berea_pnextract_ballstick_interactive_metadata.json"
DISTRIBUTION_PNG = PORE_NETWORK_DIR / "niu2020_figure4_pore_node_throat_distribution.png"
DISTRIBUTION_METADATA = PORE_NETWORK_DIR / "niu2020_figure4_pore_node_throat_distribution_metadata.json"

PNEXTRACT_INPUT_DIR = PORE_NETWORK_DIR / "pnextract_input"
PNEXTRACT_NETWORK_DIR = PORE_NETWORK_DIR / "pnextract"
PNEXTRACT_PARSED_DIR = PNEXTRACT_NETWORK_DIR / "network_parsed"

POLARIZATION_SPECTRUM_CSV = FORMAL_INPUT_DIR / "pnextract_polarization_spectrum_manual.csv"
POLARIZATION_SPECTRUM_METADATA = FORMAL_INPUT_DIR / "pnextract_polarization_spectrum_manual_metadata.json"

for path in [RUN_DIR, PORE_NETWORK_DIR, SEGMENTED_CORE_DIR, FORMAL_INPUT_DIR, COMPONENT_SPECTRA_DIR, SIM_SWEEP_DIR, FIGURE_DIR, PROVENANCE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("RUN_DIR =", RUN_DIR)
print("PNEXTRACT_NETWORK_DIR =", PNEXTRACT_NETWORK_DIR)
print("PNEXTRACT_PARSED_DIR =", PNEXTRACT_PARSED_DIR)

RUN_DIR = C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3
PNEXTRACT_NETWORK_DIR = C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\pore_network\pnextract
PNEXTRACT_PARSED_DIR = C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\pore_network\pnextract\network_parsed


In [2]:
# =========================
# 2. 通用执行函数
# =========================
def run_command(command: list[str | Path], *, cwd: Path = PROJECT_ROOT) -> None:
    command = [str(item) for item in command]
    print("\n$", " ".join(command))
    completed = subprocess.run(command, cwd=str(cwd), check=False)
    if completed.returncode != 0:
        raise RuntimeError(f"command failed with exit code {completed.returncode}: {' '.join(command)}")


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def frequency_tag(frequency_hz: float) -> str:
    if float(frequency_hz) == 1.0:
        return "1"
    if frequency_hz >= 10.0 or frequency_hz < 1.0:
        return f"{frequency_hz:.0e}".replace("+0", "").replace("+", "")
    return f"{frequency_hz:g}"


def sweep_dir_name(mechanism: str, direction: str) -> str:
    return f"manual_{mechanism}_{direction}_c128"


def canonical_checkpoint_csv(mechanism: str, direction: str, frequency_hz: float) -> Path:
    return SIM_SWEEP_DIR / f"chk_{mechanism}_{direction}_{frequency_tag(float(frequency_hz))}_c128" / "sweep_results.csv"


def manual_sweep_csv(mechanism: str, direction: str) -> Path:
    return SIM_SWEEP_DIR / sweep_dir_name(mechanism, direction) / "sweep_results.csv"


print("工具函数已加载")

工具函数已加载


## A. 重映射 CT 分割体

pnextract 和 Fiji/VTK 可视化统一使用 `pore=0, solid=255` 的派生二值体。AC3D full-grid 求解仍使用原始 RAW 和原始标签。

In [3]:
if REMAP_BINARY_VOLUME:
    import tifffile

    volume = tifffile.imread(SOURCE_SEGMENTED_TIFF)
    source_values, source_counts = np.unique(volume, return_counts=True)
    pore_values_requested = np.asarray(SOURCE_PORE_VALUES, dtype=np.int64)
    effective_pore_values = pore_values_requested.copy()
    pore_mask = np.isin(volume.astype(np.int64, copy=False), effective_pore_values)
    if not pore_mask.any() and volume.dtype == np.uint16:
        scaled_values = pore_values_requested * 256
        if np.isin(source_values.astype(np.int64), scaled_values).any():
            effective_pore_values = scaled_values
            pore_mask = np.isin(volume.astype(np.int64, copy=False), effective_pore_values)
    if not pore_mask.any():
        raise ValueError(
            f"No pore voxels matched SOURCE_PORE_VALUES={SOURCE_PORE_VALUES}; "
            f"source labels are {source_values.tolist()}"
        )
    binary = np.where(pore_mask, 0, BINARY_SOLID_VALUE).astype(np.uint8)
    tifffile.imwrite(BINARY_TIFF, binary)
    binary.tofile(BINARY_RAW)
    metadata = {
        "source_segmented_tiff": str(SOURCE_SEGMENTED_TIFF),
        "binary_tiff": str(BINARY_TIFF),
        "binary_raw": str(BINARY_RAW),
        "source_shape_zyx": [int(v) for v in volume.shape],
        "source_dtype": str(volume.dtype),
        "source_values": {str(int(v)): int(c) for v, c in zip(source_values, source_counts)},
        "pore_values": [int(v) for v in SOURCE_PORE_VALUES],
        "effective_pore_values_in_tiff": [int(v) for v in effective_pore_values],
        "solid_value_out": int(BINARY_SOLID_VALUE),
        "pore_voxels": int(np.count_nonzero(binary == 0)),
        "total_voxels": int(binary.size),
        "porosity": float(np.count_nonzero(binary == 0) / binary.size),
        "voxel_size_um": float(VOXEL_SIZE_UM),
    }
    write_json(BINARY_METADATA, metadata)
    print(json.dumps(metadata, indent=2, ensure_ascii=False))
else:
    print("跳过重映射；使用已有二值体", BINARY_TIFF)

{
  "source_segmented_tiff": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\data\\Niu 2020data\\microCT_Berea.tiff",
  "binary_tiff": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\niu2020_berea_reproduction3\\segmented_core\\microCT_Berea_solid255_pore0.tiff",
  "binary_raw": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\niu2020_berea_reproduction3\\segmented_core\\microCT_Berea_solid255_pore0.raw",
  "source_shape_zyx": [
    350,
    350,
    350
  ],
  "source_dtype": "uint16",
  "source_values": {
    "256": 9924264,
    "512": 32950736
  },
  "pore_values": [
    1
  ],
  "effective_pore_values_in_tiff": [
    256
  ],
  "solid_value_out": 255,
  "pore_voxels": 9924264,
  "total_voxels": 42875000,
  "porosity": 0.23146971428571428,
  "voxel_size_um": 2.8
}


## B. 导出 Fiji/VTK 数字岩心 HTML

输出到：`RUN_DIR/pore_network/microCT_Berea_fiji3d_fullres_volume_interactive.html`。

In [4]:
if EXPORT_FIJI3D_HTML:
    run_command([
        PYTHON_EXE,
        PROJECT_ROOT / "code" / "scripts" / "digital_rock_visualization" / "render_segmented_core_fiji3d_html.py",
        "--input", BINARY_TIFF,
        "--out", FIJI3D_HTML,
        "--metadata-out", FIJI3D_METADATA,
        "--solid-value", str(BINARY_SOLID_VALUE),
        "--downsample", str(FIJI3D_DOWNSAMPLE),
        "--voxel-size-um", str(VOXEL_SIZE_UM),
        "--initial-visible", "all",
        "--interpolation", "nearest",
    ])
else:
    print("跳过 Fiji/VTK HTML 导出")

print("Fiji/VTK HTML:", FIJI3D_HTML)


$ C:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\digital_rock_visualization\render_segmented_core_fiji3d_html.py --input C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\segmented_core\microCT_Berea_solid255_pore0.tiff --out C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\pore_network\microCT_Berea_fiji3d_fullres_volume_interactive.html --metadata-out C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\pore_network\microCT_Berea_fiji3d_fullres_volume_interactive_metadata.json --solid-value 255 --downsample 1 --voxel-size-um 2.8 --initial-visible all --interpolation nearest


Fiji/VTK HTML: C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\pore_network\microCT_Berea_fiji3d_fullres_volume_interactive.html


## C. 提取 pnextract 孔网并导出球棍 HTML + 孔径/孔喉分布图

输出都放在 `RUN_DIR/pore_network/` 下：

- `microCT_Berea_pnextract_ballstick_interactive.html`
- `niu2020_figure4_pore_node_throat_distribution.png`
- `pnextract/network_parsed/pores.csv`
- `pnextract/network_parsed/throats.csv`

In [5]:
if RUN_PNEXTRACT_AND_RENDER_NETWORK:
    run_command([
        PYTHON_EXE,
        PROJECT_ROOT / "code" / "scripts" / "pore_network" / "run_segmented_core_pnextract_ballstick.py",
        "--input", BINARY_TIFF,
        "--title", "microCT_Berea",
        "--pore-values", "0",
        "--solid-value", str(BINARY_SOLID_VALUE),
        "--voxel-size-um", str(VOXEL_SIZE_UM),
        "--downsample", str(PNEXTRACT_DOWNSAMPLE),
        "--prepare-dir", PNEXTRACT_INPUT_DIR,
        "--network-dir", PNEXTRACT_NETWORK_DIR,
        "--html-out", BALLSTICK_HTML,
        "--metadata-out", BALLSTICK_METADATA,
        "--distribution-out", DISTRIBUTION_PNG,
        "--distribution-metadata-out", DISTRIBUTION_METADATA,
        "--distribution-bins", str(DISTRIBUTION_BINS),
        "--pnextract-exe", PNEXTRACT_EXE,
    ])
else:
    print("跳过 pnextract；将复用已有网络目录", PNEXTRACT_NETWORK_DIR)

print("Ball-stick HTML:", BALLSTICK_HTML)
print("Distribution PNG:", DISTRIBUTION_PNG)
print("Parsed network dir:", PNEXTRACT_PARSED_DIR)


$ C:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\pore_network\run_segmented_core_pnextract_ballstick.py --input C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\segmented_core\microCT_Berea_solid255_pore0.tiff --title microCT_Berea --pore-values 0 --solid-value 255 --voxel-size-um 2.8 --downsample 1 --prepare-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\pore_network\pnextract_input --network-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\pore_network\pnextract --html-out C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\pore_network\microCT_Berea_pnextract_ballstick_interactive.html --metadata-out C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\pore_network\microCT_Berea_pnextract_ballstick_interactive_metadata.json --distribution-out C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\resul

Ball-stick HTML: C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\pore_network\microCT_Berea_pnextract_ballstick_interactive.html
Distribution PNG: C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\pore_network\niu2020_figure4_pore_node_throat_distribution.png
Parsed network dir: C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\pore_network\pnextract\network_parsed


## D. 自动读取孔网几何并计算极化谱

这一步读取上一步的 `pores.csv` / `throats.csv`，用孔半径、孔喉长度、孔喉半径/shape factor、几何 Zdc 计算 `delta_sigma_pore` 和 `delta_sigma_membrane`。

In [6]:
if COMPUTE_POLARIZATION_SPECTRA:
    sys.path.insert(0, str(PROJECT_ROOT / "code" / "src"))
    sys.path.insert(0, str(PROJECT_ROOT / "code" / "scripts" / "sip_simulation"))

    from pore_scale_electrical.polarization import PolarizationParameters
    from compute_polarization_spectra import compute_spectra, load_pnextract

    params = PolarizationParameters(**SIP_PARAMETERS)
    pores, throats = load_pnextract(
        PNEXTRACT_PARSED_DIR,
        params,
        include_boundary_throats=False,
    )
    spectra, metadata = compute_spectra(
        np.asarray(FREQUENCIES_HZ, dtype=float),
        pores,
        throats,
        params,
        figure5_zdc_ohm=None,
        pore_radius_scale=1.0,
        membrane_length_scale=1.0,
        membrane_zdc_scale=1.0,
        membrane_weight_mode="volume",
        membrane_zdc_length_mode="throat",
    )
    POLARIZATION_SPECTRUM_CSV.parent.mkdir(parents=True, exist_ok=True)
    spectra.to_csv(POLARIZATION_SPECTRUM_CSV, index=False)
    write_json(POLARIZATION_SPECTRUM_METADATA, metadata)
    display(spectra)
    print("wrote", POLARIZATION_SPECTRUM_CSV)
    print("wrote", POLARIZATION_SPECTRUM_METADATA)
else:
    print("跳过极化谱计算；使用已有文件", POLARIZATION_SPECTRUM_CSV)

,frequency_hz,omega_rad_s,pore_conductance_real_s,pore_conductance_imag_s,membrane_conductance_real_s,membrane_conductance_imag_s,total_conductance_real_s,total_conductance_imag_s,delta_sigma_pore_real_s_m,delta_sigma_pore_imag_s_m,...,delta_sigma_membrane_imag_s_m,delta_sigma_total_real_s_m,delta_sigma_total_imag_s_m,apparent_water_sigma_real_s_m,apparent_water_sigma_imag_s_m,membrane_effective_length_m_mean,membrane_relaxation_length_m_mean,membrane_zdc_length_m_mean,membrane_active_area_m2_mean,membrane_effective_zdc_ohm_mean
0,1.000000e-03,6.283185e-03,5.361759e-15,2.215004e-12,2.124196e-10,2.036527e-10,2.124250e-10,2.058677e-10,3.971674e-09,1.640744e-06,...,1.508539e-04,0.000157,1.524946e-04,0.043157,0.000152,0.000028,0.000028,0.000028,4.079686e-10,9.885991e+06
1,1.000000e-02,6.283185e-02,5.356779e-13,2.213473e-11,6.675925e-10,5.864659e-10,6.681282e-10,6.086006e-10,3.967984e-07,1.639609e-05,...,4.344192e-04,0.000495,4.508153e-04,0.043495,0.000451,0.000028,0.000028,0.000028,4.079686e-10,9.885991e+06
2,1.000000e-01,6.283185e-01,4.917022e-11,2.075751e-10,2.007525e-09,1.373521e-09,2.056695e-09,1.581096e-09,3.642239e-05,1.537593e-04,...,1.017423e-03,0.001523,1.171182e-03,0.044523,0.001171,0.000028,0.000028,0.000028,4.079686e-10,9.885991e+06
3,1.000000e+00,6.283185e+00,7.893904e-10,4.823072e-10,4.758961e-09,1.734646e-09,5.548351e-09,2.216954e-09,5.847336e-04,3.572646e-04,...,1.284923e-03,0.004110,1.642188e-03,0.047110,0.001642,0.000028,0.000028,0.000028,4.079686e-10,9.885991e+06
4,1.000000e+03,6.283185e+03,1.299989e-09,1.894988e-12,7.373808e-09,7.361297e-11,8.673797e-09,7.550795e-11,9.629550e-04,1.403695e-06,...,5.452812e-05,0.006425,5.593182e-05,0.049425,0.000060,0.000028,0.000028,0.000028,4.079686e-10,9.885991e+06
5,1.000000e+06,6.283185e+06,1.300000e-09,1.895124e-15,7.445098e-09,2.327998e-12,8.745098e-09,2.329893e-12,9.629630e-04,1.403796e-09,...,1.724443e-06,0.006478,1.725846e-06,0.049478,0.004450,0.000028,0.000028,0.000028,4.079686e-10,9.885991e+06
6,1.000000e+09,6.283185e+09,1.300000e-09,1.895124e-18,7.447352e-09,7.361790e-14,8.747352e-09,7.361979e-14,9.629630e-04,1.403796e-12,...,5.453178e-08,0.006480,5.453318e-08,0.049480,4.448495,0.000028,0.000028,0.000028,4.079686e-10,9.885991e+06


wrote C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\formal_inputs\pnextract_polarization_spectrum_manual.csv
wrote C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\formal_inputs\pnextract_polarization_spectrum_manual_metadata.json


## E. 按 Niu Section 5.3 生成分机制 AC3D 输入谱

这里直接使用上面 `SIP_PARAMETERS`，不会退回脚本里的默认参数。

In [7]:
def make_component_spectrum(base: pd.DataFrame, component: str, params, mode: str = "paper") -> pd.DataFrame:
    omega = base["omega_rad_s"].to_numpy(dtype=float)
    water_dielectric_imag = omega * params.water_permittivity_f_m
    solid_dielectric_imag = omega * params.solid_permittivity_f_m

    if component == "interfacial":
        delta_real = np.zeros(len(base), dtype=float)
        delta_imag = np.zeros(len(base), dtype=float)
    elif component == "pore":
        delta_real = base["delta_sigma_pore_real_s_m"].to_numpy(dtype=float)
        delta_imag = base["delta_sigma_pore_imag_s_m"].to_numpy(dtype=float)
    elif component == "membrane":
        delta_real = base["delta_sigma_membrane_real_s_m"].to_numpy(dtype=float)
        delta_imag = base["delta_sigma_membrane_imag_s_m"].to_numpy(dtype=float)
    elif component == "all":
        delta_real = base["delta_sigma_total_real_s_m"].to_numpy(dtype=float)
        delta_imag = base["delta_sigma_total_imag_s_m"].to_numpy(dtype=float)
    else:
        raise ValueError(component)

    if mode == "paper" and component in {"pore", "membrane"}:
        water_real = params.water_conductivity_s_m + delta_real
        water_imag = delta_imag
        solid_real = np.zeros(len(base), dtype=float)
        solid_imag = np.zeros(len(base), dtype=float)
    else:
        water_real = params.water_conductivity_s_m + delta_real
        water_imag = water_dielectric_imag + delta_imag
        solid_real = np.zeros(len(base), dtype=float)
        solid_imag = solid_dielectric_imag

    return pd.DataFrame({
        "frequency_hz": base["frequency_hz"].to_numpy(dtype=float),
        "omega_rad_s": omega,
        "component": component,
        "component_mode": mode,
        "delta_sigma_component_real_s_m": delta_real,
        "delta_sigma_component_imag_s_m": delta_imag,
        "apparent_water_sigma_real_s_m": water_real,
        "apparent_water_sigma_imag_s_m": water_imag,
        "solid_sigma_real_s_m": solid_real,
        "solid_sigma_imag_s_m": solid_imag,
    })


if MAKE_COMPONENT_SPECTRA:
    sys.path.insert(0, str(PROJECT_ROOT / "code" / "src"))
    from pore_scale_electrical.polarization import PolarizationParameters

    params = PolarizationParameters(**SIP_PARAMETERS)
    base = pd.read_csv(POLARIZATION_SPECTRUM_CSV)
    COMPONENT_SPECTRA_DIR.mkdir(parents=True, exist_ok=True)
    component_paths = {}
    for mechanism in MECHANISMS:
        out = make_component_spectrum(base, mechanism, params, mode="paper")
        path = COMPONENT_SPECTRA_DIR / f"polarization_spectra_{mechanism}.csv"
        out.to_csv(path, index=False)
        component_paths[mechanism] = str(path)
    write_json(COMPONENT_SPECTRA_DIR / "metadata.json", {
        "source": str(POLARIZATION_SPECTRUM_CSV),
        "mode": "paper",
        "parameters": SIP_PARAMETERS,
        "definition": {
            "interfacial": "water dc conductivity plus water/solid high-frequency permittivity only; no pore or membrane increment",
            "pore": "water sigma_w plus delta_sigma_pore only; solid phase is zero",
            "membrane": "water sigma_w plus delta_sigma_membrane only; solid phase is zero",
            "all": "water sigma_w plus dielectric plus pore and membrane increments; solid phase has dielectric term",
        },
        **component_paths,
    })
    print(json.dumps(component_paths, indent=2, ensure_ascii=False))
else:
    print("跳过机制谱生成；使用已有目录", COMPONENT_SPECTRA_DIR)

{
  "interfacial": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\niu2020_berea_reproduction3\\source_data\\formal_inputs\\component_spectra_manual\\polarization_spectra_interfacial.csv",
  "pore": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\niu2020_berea_reproduction3\\source_data\\formal_inputs\\component_spectra_manual\\polarization_spectra_pore.csv",
  "membrane": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\niu2020_berea_reproduction3\\source_data\\formal_inputs\\component_spectra_manual\\polarization_spectra_membrane.csv",
  "all": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\niu2020_berea_reproduction3\\source_data\\formal_inputs\\component_spectra_manual\\polarization_spectra_all.csv"
}


## F. 可选：运行 full-grid AC3D 分机制 x/y/z sweep

这个步骤很重。默认 `RUN_FULLGRID_AC3D = False`。如果你要完整手动重跑，把第一个代码单元里该变量改成 `True`。

In [8]:
if RUN_FULLGRID_AC3D:
    for mechanism in MECHANISMS:
        spectra_csv = COMPONENT_SPECTRA_DIR / f"polarization_spectra_{mechanism}.csv"
        for direction in DIRECTIONS:
            out_dir = SIM_SWEEP_DIR / sweep_dir_name(mechanism, direction)
            run_command([
                PYTHON_EXE,
                PROJECT_ROOT / "code" / "scripts" / "sip_simulation" / "run_ac3d_matrix_free_gpu_sweep.py",
                "--raw", BINARY_TIFF,
                "--shape", *[str(v) for v in AC3D_SHAPE_ZYX],
                "--pore-label", str(AC3D_PORE_LABEL),
                "--solid-label", str(AC3D_SOLID_LABEL),
                "--voxel-size-m", str(VOXEL_SIZE_M),
                "--spectra", spectra_csv,
                "--frequencies", *[f"{v:g}" for v in FREQUENCIES_HZ],
                "--frequency-match-mode", "exact",
                "--direction", direction,
                "--dtype", AC3D_DTYPE,
                "--preconditioner", AC3D_PRECONDITIONER,
                "--fft-reference", AC3D_FFT_REFERENCE,
                "--gauge-mode", AC3D_GAUGE_MODE,
                "--rtol", AC3D_RTOL,
                "--atol", AC3D_ATOL,
                "--maxiter", AC3D_MAXITER,
                "--residual-every", AC3D_RESIDUAL_EVERY,
                "--progress-every", AC3D_RESIDUAL_EVERY,
                "--resume",
                "--out-dir", out_dir,
            ])
else:
    print("未运行 full-grid AC3D。若要完整手动重跑，请设置 RUN_FULLGRID_AC3D = True。")


$ C:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\sip_simulation\run_ac3d_matrix_free_gpu_sweep.py --raw C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\segmented_core\microCT_Berea_solid255_pore0.tiff --shape 350 350 350 --pore-label 0 --solid-label 255 --voxel-size-m 2.8e-06 --spectra C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\formal_inputs\component_spectra_manual\polarization_spectra_interfacial.csv --frequencies 0.001 0.01 0.1 1 1000 1e+06 1e+09 --frequency-match-mode exact --direction x --dtype complex128 --preconditioner fft --fft-reference pore --gauge-mode auto --rtol 1e-5 --atol 0 --maxiter 1000 --residual-every 50 --progress-every 50 --resume --out-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\simulation_sweeps\manual_interfacial_x_c128



$ C:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\sip_simulation\run_ac3d_matrix_free_gpu_sweep.py --raw C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\segmented_core\microCT_Berea_solid255_pore0.tiff --shape 350 350 350 --pore-label 0 --solid-label 255 --voxel-size-m 2.8e-06 --spectra C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\formal_inputs\component_spectra_manual\polarization_spectra_interfacial.csv --frequencies 0.001 0.01 0.1 1 1000 1e+06 1e+09 --frequency-match-mode exact --direction y --dtype complex128 --preconditioner fft --fft-reference pore --gauge-mode auto --rtol 1e-5 --atol 0 --maxiter 1000 --residual-every 50 --progress-every 50 --resume --out-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\simulation_sweeps\manual_interfacial_y_c128



$ C:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\sip_simulation\run_ac3d_matrix_free_gpu_sweep.py --raw C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\segmented_core\microCT_Berea_solid255_pore0.tiff --shape 350 350 350 --pore-label 0 --solid-label 255 --voxel-size-m 2.8e-06 --spectra C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\formal_inputs\component_spectra_manual\polarization_spectra_interfacial.csv --frequencies 0.001 0.01 0.1 1 1000 1e+06 1e+09 --frequency-match-mode exact --direction z --dtype complex128 --preconditioner fft --fft-reference pore --gauge-mode auto --rtol 1e-5 --atol 0 --maxiter 1000 --residual-every 50 --progress-every 50 --resume --out-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\simulation_sweeps\manual_interfacial_z_c128



$ C:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\sip_simulation\run_ac3d_matrix_free_gpu_sweep.py --raw C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\segmented_core\microCT_Berea_solid255_pore0.tiff --shape 350 350 350 --pore-label 0 --solid-label 255 --voxel-size-m 2.8e-06 --spectra C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\formal_inputs\component_spectra_manual\polarization_spectra_pore.csv --frequencies 0.001 0.01 0.1 1 1000 1e+06 1e+09 --frequency-match-mode exact --direction x --dtype complex128 --preconditioner fft --fft-reference pore --gauge-mode auto --rtol 1e-5 --atol 0 --maxiter 1000 --residual-every 50 --progress-every 50 --resume --out-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\simulation_sweeps\manual_pore_x_c128



$ C:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\sip_simulation\run_ac3d_matrix_free_gpu_sweep.py --raw C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\segmented_core\microCT_Berea_solid255_pore0.tiff --shape 350 350 350 --pore-label 0 --solid-label 255 --voxel-size-m 2.8e-06 --spectra C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\formal_inputs\component_spectra_manual\polarization_spectra_pore.csv --frequencies 0.001 0.01 0.1 1 1000 1e+06 1e+09 --frequency-match-mode exact --direction y --dtype complex128 --preconditioner fft --fft-reference pore --gauge-mode auto --rtol 1e-5 --atol 0 --maxiter 1000 --residual-every 50 --progress-every 50 --resume --out-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\simulation_sweeps\manual_pore_y_c128


...


$ C:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\sip_simulation\run_ac3d_matrix_free_gpu_sweep.py --raw C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\segmented_core\microCT_Berea_solid255_pore0.tiff --shape 350 350 350 --pore-label 0 --solid-label 255 --voxel-size-m 2.8e-06 --spectra C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\formal_inputs\component_spectra_manual\polarization_spectra_membrane.csv --frequencies 0.001 0.01 0.1 1 1000 1e+06 1e+09 --frequency-match-mode exact --direction y --dtype complex128 --preconditioner fft --fft-reference pore --gauge-mode auto --rtol 1e-5 --atol 0 --maxiter 1000 --residual-every 50 --progress-every 50 --resume --out-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\simulation_sweeps\manual_membrane_y_c128



$ C:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\sip_simulation\run_ac3d_matrix_free_gpu_sweep.py --raw C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\segmented_core\microCT_Berea_solid255_pore0.tiff --shape 350 350 350 --pore-label 0 --solid-label 255 --voxel-size-m 2.8e-06 --spectra C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\formal_inputs\component_spectra_manual\polarization_spectra_membrane.csv --frequencies 0.001 0.01 0.1 1 1000 1e+06 1e+09 --frequency-match-mode exact --direction z --dtype complex128 --preconditioner fft --fft-reference pore --gauge-mode auto --rtol 1e-5 --atol 0 --maxiter 1000 --residual-every 50 --progress-every 50 --resume --out-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\simulation_sweeps\manual_membrane_z_c128



$ C:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\sip_simulation\run_ac3d_matrix_free_gpu_sweep.py --raw C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\segmented_core\microCT_Berea_solid255_pore0.tiff --shape 350 350 350 --pore-label 0 --solid-label 255 --voxel-size-m 2.8e-06 --spectra C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\formal_inputs\component_spectra_manual\polarization_spectra_all.csv --frequencies 0.001 0.01 0.1 1 1000 1e+06 1e+09 --frequency-match-mode exact --direction x --dtype complex128 --preconditioner fft --fft-reference pore --gauge-mode auto --rtol 1e-5 --atol 0 --maxiter 1000 --residual-every 50 --progress-every 50 --resume --out-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\simulation_sweeps\manual_all_x_c128



$ C:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\sip_simulation\run_ac3d_matrix_free_gpu_sweep.py --raw C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\segmented_core\microCT_Berea_solid255_pore0.tiff --shape 350 350 350 --pore-label 0 --solid-label 255 --voxel-size-m 2.8e-06 --spectra C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\formal_inputs\component_spectra_manual\polarization_spectra_all.csv --frequencies 0.001 0.01 0.1 1 1000 1e+06 1e+09 --frequency-match-mode exact --direction y --dtype complex128 --preconditioner fft --fft-reference pore --gauge-mode auto --rtol 1e-5 --atol 0 --maxiter 1000 --residual-every 50 --progress-every 50 --resume --out-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\simulation_sweeps\manual_all_y_c128



$ C:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\sip_simulation\run_ac3d_matrix_free_gpu_sweep.py --raw C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\segmented_core\microCT_Berea_solid255_pore0.tiff --shape 350 350 350 --pore-label 0 --solid-label 255 --voxel-size-m 2.8e-06 --spectra C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\formal_inputs\component_spectra_manual\polarization_spectra_all.csv --frequencies 0.001 0.01 0.1 1 1000 1e+06 1e+09 --frequency-match-mode exact --direction z --dtype complex128 --preconditioner fft --fft-reference pore --gauge-mode auto --rtol 1e-5 --atol 0 --maxiter 1000 --residual-every 50 --progress-every 50 --resume --out-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\simulation_sweeps\manual_all_z_c128


## G. 汇总方向平均

如果当前结果包里已经有这次完成的 canonical checkpoint sweep，会优先复用 `chk_<mechanism>_<direction>_<freq>_c128`。否则使用 notebook 手动跑出的 `manual_<mechanism>_<direction>_c128/sweep_results.csv`。

In [9]:
def load_mechanism_direction_frame(mechanism: str, direction: str) -> pd.DataFrame:
    manual_csv = manual_sweep_csv(mechanism, direction)
    if manual_csv.exists():
        return pd.read_csv(manual_csv)

    frames = []
    for frequency in FREQUENCIES_HZ:
        path = canonical_checkpoint_csv(mechanism, direction, frequency)
        if not path.exists():
            raise FileNotFoundError(f"缺少 {mechanism}/{direction}/{frequency:g} Hz sweep: {path}")
        frames.append(pd.read_csv(path))
    return pd.concat(frames, ignore_index=True).sort_values("frequency_hz").reset_index(drop=True)


def summarize_mechanism(mechanism: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    frames = {direction: load_mechanism_direction_frame(mechanism, direction) for direction in DIRECTIONS}
    frequency = frames["x"]["frequency_hz"].to_numpy(dtype=float)
    for direction in DIRECTIONS[1:]:
        other = frames[direction]["frequency_hz"].to_numpy(dtype=float)
        if not np.array_equal(frequency, other):
            raise ValueError(f"{mechanism}: x/y/z frequency grids differ")
    real = np.column_stack([frames[d]["effective_sigma_real_s_m"].to_numpy(dtype=float) for d in DIRECTIONS])
    imag = np.column_stack([frames[d]["effective_sigma_imag_s_m"].to_numpy(dtype=float) for d in DIRECTIONS])
    residual = np.column_stack([frames[d].get("true_residual_norm", pd.Series(np.nan, index=frames[d].index)).to_numpy(dtype=float) for d in DIRECTIONS])
    detail = pd.DataFrame({
        "mechanism": mechanism,
        "frequency_hz": frequency,
        "sigma_xx_real_s_m": real[:, 0],
        "sigma_yy_real_s_m": real[:, 1],
        "sigma_zz_real_s_m": real[:, 2],
        "sigma_xx_imag_s_m": imag[:, 0],
        "sigma_yy_imag_s_m": imag[:, 1],
        "sigma_zz_imag_s_m": imag[:, 2],
        "directional_mean_real_s_m": real.mean(axis=1),
        "directional_mean_imag_s_m": imag.mean(axis=1),
        "anisotropy_ratio_real": np.nanmax(real, axis=1) / np.maximum(np.abs(np.nanmin(real, axis=1)), np.finfo(float).eps),
        "anisotropy_ratio_imag": np.nanmax(np.abs(imag), axis=1) / np.maximum(np.nanmin(np.abs(imag), axis=1), np.finfo(float).eps),
        "max_true_residual_norm": np.nanmax(residual, axis=1),
    })
    plot_ready = pd.DataFrame({
        "frequency_hz": detail["frequency_hz"],
        "effective_sigma_real_s_m": detail["directional_mean_real_s_m"],
        "effective_sigma_imag_s_m": detail["directional_mean_imag_s_m"],
        "max_true_residual_norm": detail["max_true_residual_norm"],
    })
    return detail, plot_ready


if SUMMARIZE_AND_PLOT:
    all_detail = []
    plot_paths = {}
    for mechanism in MECHANISMS:
        detail, plot_ready = summarize_mechanism(mechanism)
        detail_path = RUN_DIR / "source_data" / f"manual_{mechanism}_directional_summary.csv"
        plot_path = RUN_DIR / "source_data" / f"manual_{mechanism}_directional_mean_sweep.csv"
        detail.to_csv(detail_path, index=False)
        plot_ready.to_csv(plot_path, index=False)
        all_detail.append(detail)
        plot_paths[mechanism] = plot_path
    combined = pd.concat(all_detail, ignore_index=True)
    combined_path = RUN_DIR / "source_data" / "manual_mechanism_directional_summary.csv"
    combined.to_csv(combined_path, index=False)
    write_json(PROVENANCE_DIR / "manual_mechanism_directional_summary.json", {
        "combined_csv": str(combined_path),
        "plot_ready_csvs": {k: str(v) for k, v in plot_paths.items()},
        "max_true_residual_norm": float(combined["max_true_residual_norm"].max()),
        "frequencies_hz": FREQUENCIES_HZ,
        "mechanisms": MECHANISMS,
        "directions": DIRECTIONS,
    })
    display(combined)
else:
    print("跳过方向汇总")

,mechanism,frequency_hz,sigma_xx_real_s_m,sigma_yy_real_s_m,sigma_zz_real_s_m,sigma_xx_imag_s_m,sigma_yy_imag_s_m,sigma_zz_imag_s_m,directional_mean_real_s_m,directional_mean_imag_s_m,anisotropy_ratio_real,anisotropy_ratio_imag,max_true_residual_norm
0,interfacial,1.000000e-03,0.002620,0.001830,0.002014,1.042820e-09,4.559595e-10,2.969867e-09,0.002155,1.489549e-09,1.431907,6.513444,0.000009
1,interfacial,1.000000e-02,0.002620,0.001830,0.002014,1.048726e-09,4.611549e-10,2.975217e-09,0.002155,1.495033e-09,1.431907,6.451665,0.000009
2,interfacial,1.000000e-01,0.002620,0.001830,0.002014,1.107792e-09,5.131091e-10,3.028722e-09,0.002155,1.549874e-09,1.431907,5.902685,0.000009
3,interfacial,1.000000e+00,0.002620,0.001830,0.002014,1.698446e-09,1.032651e-09,3.563768e-09,0.002155,2.098288e-09,1.431907,3.451086,0.000009
4,interfacial,1.000000e+03,0.002620,0.001830,0.002014,1.796962e-06,1.883619e-06,1.996538e-06,0.002155,1.892373e-06,1.431922,1.111063,0.000010
5,interfacial,1.000000e+06,0.002900,0.002164,0.002290,1.250106e-03,1.296727e-03,1.213386e-03,0.002451,1.253406e-03,1.340115,1.068684,0.000010
6,interfacial,1.000000e+09,0.003912,0.003358,0.003348,8.726544e-01,8.330503e-01,8.284543e-01,0.003540,8.447197e-01,1.168434,1.053353,0.000009
7,pore,1.000000e-03,0.002620,0.001830,0.002014,1.046445e-07,6.759514e-08,9.608178e-08,0.002155,8.944049e-08,1.431948,1.548108,0.000010
8,pore,1.000000e-02,0.002620,0.001830,0.002014,1.003720e-06,6.954641e-07,7.871936e-07,0.002155,8.287927e-07,1.431948,1.443238,0.000010
9,pore,1.000000e-01,0.002622,0.001831,0.002016,9.373567e-06,6.540536e-06,7.221034e-06,0.002156,7.711712e-06,1.431948,1.433150,0.000010


## H. 导出 SIP 机制结果图

默认使用 Niu `Figure7.xlsx` / `Figure8.xlsx` 实验数据做对照；如果换成自己的实验数据，需要改绘图脚本或准备同格式输入。

In [10]:
if SUMMARIZE_AND_PLOT:
    run_command([
        PYTHON_EXE,
        PROJECT_ROOT / "code" / "scripts" / "sip_simulation" / "plot_niu2020_conductivity_mechanism_comparison.py",
        "--data-dir", PROJECT_ROOT / "data" / "Niu 2020data",
        "--all-csv", RUN_DIR / "source_data" / "manual_all_directional_mean_sweep.csv",
        "--pore-csv", RUN_DIR / "source_data" / "manual_pore_directional_mean_sweep.csv",
        "--membrane-csv", RUN_DIR / "source_data" / "manual_membrane_directional_mean_sweep.csv",
        "--interfacial-csv", RUN_DIR / "source_data" / "manual_interfacial_directional_mean_sweep.csv",
        "--figure-base", FIGURE_DIR / "manual_sip_mechanism_comparison",
        "--source-data-csv", RUN_DIR / "source_data" / "manual_sip_mechanism_comparison_source_data.csv",
        "--summary-json", PROVENANCE_DIR / "manual_sip_mechanism_comparison_summary.json",
        "--provenance-md", PROVENANCE_DIR / "manual_sip_mechanism_comparison_provenance.md",
        "--title", "Manual SIP full-grid directional mean",
    ])
    print("Figure PNG:", FIGURE_DIR / "manual_sip_mechanism_comparison.png")
else:
    print("跳过绘图")


$ C:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\sip_simulation\plot_niu2020_conductivity_mechanism_comparison.py --data-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\data\Niu 2020data --all-csv C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\manual_all_directional_mean_sweep.csv --pore-csv C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\manual_pore_directional_mean_sweep.csv --membrane-csv C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\manual_membrane_directional_mean_sweep.csv --interfacial-csv C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\source_data\manual_interfacial_directional_mean_sweep.csv --figure-base C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\figures\manual_sip_mechanism_comparison --source-data-csv C:\Users\imgw\Documents\Codex\SIP模拟\

Figure PNG: C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\figures\manual_sip_mechanism_comparison.png


## I. 最终验收检查

正式结果至少要求：`frequency_match_mode=exact`、`info=0`、`true_residual_passed=True`。

In [11]:
rows = []
for mechanism in MECHANISMS:
    for direction in DIRECTIONS:
        frame = load_mechanism_direction_frame(mechanism, direction)
        rows.append({
            "mechanism": mechanism,
            "direction": direction,
            "n_frequency": int(len(frame)),
            "all_info_zero": bool((frame["info"].astype(int) == 0).all()) if "info" in frame else None,
            "all_true_residual_passed": bool(frame["true_residual_passed"].astype(bool).all()) if "true_residual_passed" in frame else None,
            "max_true_residual_norm": float(frame["true_residual_norm"].max()) if "true_residual_norm" in frame else np.nan,
            "all_exact_frequency": bool(frame["frequency_match_mode"].astype(str).eq("exact").all()) if "frequency_match_mode" in frame else None,
        })

qa = pd.DataFrame(rows)
qa_path = PROVENANCE_DIR / "manual_sip_workflow_acceptance_summary.csv"
qa.to_csv(qa_path, index=False)
display(qa)
print("wrote", qa_path)

,mechanism,direction,n_frequency,all_info_zero,all_true_residual_passed,max_true_residual_norm,all_exact_frequency
0,interfacial,x,7,True,True,0.000010,True
1,interfacial,y,7,True,True,0.000010,True
2,interfacial,z,7,True,True,0.000010,True
3,pore,x,7,True,True,0.000010,True
4,pore,y,7,True,True,0.000007,True
5,pore,z,7,True,True,0.000010,True
6,membrane,x,7,True,True,0.000005,True
7,membrane,y,7,True,True,0.000009,True
8,membrane,z,7,True,True,0.000004,True
9,all,x,7,True,True,0.000010,True


wrote C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\niu2020_berea_reproduction3\provenance\manual_sip_workflow_acceptance_summary.csv
